In [2]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.autograd as autograd
import matplotlib.pyplot as plt
from pylab import rcParams
rcParams['figure.figsize'] = (5, 3)

from functools import partial
# print() helper that adds two line breaks (for readability)
printn = partial(print, end='\n\n')

seed = 1234
np.random.seed(seed)

In [3]:
# Forward pass
# 4行4列のランダムな入力データxを生成
x = torch.randn(4, 4)
# 4行1列のランダムな正解ラベルyを生成
y = torch.randn(4, 1)

# 重みを初期化
# requires_grad=True：PyTorchはこの変数に対して勾配を記録・計算するようになる
w = torch.randn(4, 1, requires_grad=True)
# バイアスの初期化
b = torch.randn(1, requires_grad=True)

# 予測値yを計算
y_pred = torch.matmul(x, w) + b

# Define the objective function
# 損失を計算
# 予測値と正解の差を取ってその合計を出す「二乗誤差」を計算
loss = (y_pred - y).pow(2).sum()

In [4]:
x = torch.randn(4, 4)
y = torch.randn(4, 1)

w = torch.randn(4, 1, requires_grad=True)
# バイアスを勾配計算する対象として宣言
b = torch.randn(1, requires_grad=True)
# b.detach()：計算グラフからbえお切り離す。これ以降、bは勾配を計算しなくてもいいただの数値のとして扱われる
b = b.detach()  # stop computing gradients for b

y_pred = torch.matmul(x, w) + b

loss = (y_pred - y).pow(2).sum()

loss.backward()

print(w.grad)  # has gradients
print(b.grad)  # has no gradients

tensor([[-1.7049],
        [ 3.3385],
        [19.5500],
        [ 0.5103]])
None


In [5]:
a = nn.Sigmoid()(torch.tensor([2]))
print(a)

a = F.sigmoid(torch.tensor([2]))
print(a)

tensor([0.8808])
tensor([0.8808])


In [6]:
# By default, a tensor does not require gradients
a = torch.zeros(1)
print(a.requires_grad)

# Wrapping with nn.Parameter makes it a learnable parameter (requires gradients)
a = nn.Parameter(a)
print(a.requires_grad)

# Turn off gradient computation for this tensor
a.requires_grad = False
print(a.requires_grad)

False
True
False


In [7]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import torch.autograd as autograd
import matplotlib.pyplot as plt
from pylab import rcParams
rcParams['figure.figsize'] = (5, 3)

from functools import partial
# print() helper that adds two line breaks (for readability)
printn = partial(print, end='\n\n')

seed = 1234
np.random.seed(seed)

In [13]:
def relu(x):
    x = torch.where(x > 0, x, torch.zeros_like(x))
    return x

def softmax(x):
    x -= torch.cat([x.max(axis=1, keepdim=True).values] * x.size()[1], dim=1)
    x_exp = torch.exp(x)
    return x_exp/torch.cat([x_exp.sum(dim=1, keepdim=True)] * x.size()[1], dim=1)


class Dense(nn.Module):  # inherit from nn.Module
    def __init__(self, in_dim, out_dim, function=lambda x: x):
        super().__init__()
        # He Initialization
        # in_dim: input dimension, out_dim: output dimension
        # 重みを初期化
        # np.sqrt(6/in_dim)：Heの初期値
        # nn.Parameter：self.Wとself.bは学習させる対象（更新するパラメータ）だとPyTorchに教える
        self.W = nn.Parameter(torch.tensor(np.random.uniform(
                        low=-np.sqrt(6/in_dim),
                        high=np.sqrt(6/in_dim),
                        size=(in_dim, out_dim)
                    ).astype('float32')))
        self.b = nn.Parameter(torch.tensor(np.zeros([out_dim]).astype('float32')))
        self.function = function

    def forward(self, x):  # override forward
        return self.function(torch.matmul(x, self.W) + self.b)

In [14]:
from torchvision import datasets, transforms

In [ ]:
# MNISTデータセットを用いたPyTorchの画像認識モデルを作成
class MLP(nn.Module):
    def __init__(self, in_dim, hid_dim, out_dim):
        super(MLP, self).__init__()
        self.linear1 = Dense(in_dim, hid_dim, function=relu) # WRITE ME
        self.linear2 = Dense(hid_dim, out_dim, function=softmax) # WRITE ME

    def forward(self, x):
        x = self.linear1(x) # WRITE ME
        x = self.linear2(x) # WRITE ME
        return x

In [16]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [17]:
in_dim = 784
hid_dim = 200
out_dim = 10
lr = 0.001
batch_size = 32
n_epochs = 10


mlp = MLP(in_dim, hid_dim, out_dim).to(device)

optimizer = optim.SGD(mlp.parameters(), lr=lr)

In [18]:
# Define preprocessing
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x.view(in_dim))
])

# Load MNIST using torchvision.datasets
# Define a DataLoader that handles mini-batching and preprocessing
dataloader_train = torch.utils.data.DataLoader(
    datasets.MNIST('../data/mnist', train=True, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=True
)

dataloader_valid = torch.utils.data.DataLoader(
    datasets.MNIST('../data/mnist', train=False, download=True, transform=transform),
    batch_size=batch_size,
    shuffle=False
)

100%|██████████| 9.91M/9.91M [00:06<00:00, 1.42MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 153kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.10MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 7.32MB/s]


In [19]:
for epoch in range(n_epochs):
    losses_train = []
    losses_valid = []
    train_num = 0
    train_true_num = 0
    valid_num = 0
    valid_true_num = 0

    mlp.train()  # Training mode (compute gradients)
    for x, t in dataloader_train:
        # Convert labels to one-hot vectors
        t_hot = torch.eye(10)[t]  # WRITE ME

        # Move tensors to GPU
        x = x.to(device)  # WRITE ME
        t_hot = t_hot.to(device)  # WRITE ME

        # Forward pass
        y = mlp(x)  # WRITE ME

        # Compute loss (cross-entropy)
        loss = -(t_hot * torch.log(y)).sum(dim=1).mean()  # WRITE ME

        # Backpropagate the loss
        optimizer.zero_grad()  # WRITE ME
        loss.backward()  # WRITE ME

        # Update parameters
        optimizer.step()  # WRITE ME
        # Convert the model output to scalar class predictions
        pred = y.argmax(1)  # WRITE ME

        losses_train.append(loss.tolist())

        acc = torch.where(t - pred.to("cpu") == 0, torch.ones_like(t), torch.zeros_like(t))
        train_num += acc.size()[0]
        train_true_num += acc.sum().item()

    mlp.eval()  # Switch to eval mode (do not compute gradients during evaluation)
    for x, t in dataloader_valid:
        t_hot = torch.eye(10)[t]  # Convert labels to one-hot vectors

        # Move tensors to GPU
        x = x.to(device)
        t_hot = t_hot.to(device)

        # Forward pass
        y = mlp(x)

        # Compute loss (cross-entropy)
        loss = -(t_hot * torch.log(y)).sum(dim=1).mean()

        # Convert the model output to scalar class predictions
        pred = y.argmax(1)

        losses_valid.append(loss.tolist())

        acc = torch.where(t - pred.to("cpu") == 0, torch.ones_like(t), torch.zeros_like(t))
        valid_num += acc.size()[0]
        valid_true_num += acc.sum().item()

    print('EPOCH: {}, Train [Loss: {:.3f}, Accuracy: {:.3f}], Valid [Loss: {:.3f}, Accuracy: {:.3f}]'.format(
        epoch,
        np.mean(losses_train),
        train_true_num / train_num,
        np.mean(losses_valid),
        valid_true_num / valid_num
    ))

EPOCH: 0, Train [Loss: 1.607, Accuracy: 0.601], Valid [Loss: 1.079, Accuracy: 0.789]
EPOCH: 1, Train [Loss: 0.878, Accuracy: 0.816], Valid [Loss: 0.699, Accuracy: 0.847]
EPOCH: 2, Train [Loss: 0.644, Accuracy: 0.852], Valid [Loss: 0.557, Accuracy: 0.871]
EPOCH: 3, Train [Loss: 0.541, Accuracy: 0.867], Valid [Loss: 0.485, Accuracy: 0.882]
EPOCH: 4, Train [Loss: 0.483, Accuracy: 0.877], Valid [Loss: 0.440, Accuracy: 0.889]
EPOCH: 5, Train [Loss: 0.445, Accuracy: 0.884], Valid [Loss: 0.410, Accuracy: 0.893]
EPOCH: 6, Train [Loss: 0.418, Accuracy: 0.889], Valid [Loss: 0.388, Accuracy: 0.897]
EPOCH: 7, Train [Loss: 0.397, Accuracy: 0.893], Valid [Loss: 0.371, Accuracy: 0.900]
EPOCH: 8, Train [Loss: 0.381, Accuracy: 0.896], Valid [Loss: 0.357, Accuracy: 0.902]
EPOCH: 9, Train [Loss: 0.368, Accuracy: 0.899], Valid [Loss: 0.346, Accuracy: 0.904]
